# Fine-Tuning LLM with Unsloth (CUDA/NVIDIA GPU)
This notebook demonstrates how to fine-tune a language model using **Unsloth** on NVIDIA GPUs with CUDA.

**Advantages of Unsloth:**
- 2x faster than normal fine-tuning
- 40% less memory usage
- 0% accuracy degradation
- Compatible with the Hugging Face ecosystem (TRL, PEFT, Transformers)

**Requirements:**
- NVIDIA GPU (GTX 1070+, RTX series, A100, H100)
- CUDA installed
- Python 3.8+

---

## 1. Setup Environment

Let's install Unsloth and the necessary libraries. Unsloth is optimized for CUDA and supports Llama and Mistral architectures.

In [1]:
!pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
!pip install --no-deps unsloth

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.4/43.4 MB 16.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.4/59.4 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 465.5/465.5 kB 26.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.6/289.6 kB 18.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.6/64.6 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 359.3/359.3 kB 18.6 MB/s eta 0:00:00


### Check CUDA availability

In [ ]:
import torch
import logging

logging.basicConfig(level=logging.INFO)


logging.info(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    logging.info(f"GPU: {torch.cuda.get_device_name(0)}")
    logging.info(
        f"Memoria GPU: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
    logging.info(f"CUDA Version: {torch.version.cuda}")

## 2. Model Configuration

Let's load the model using `FastLanguageModel.from_pretrained`. This method automatically applies the Unsloth optimizations and returns both the model and the tokenizer. 

Unsloth supports pre-quantized 4-bit models that further reduce memory usage.

In [ ]:
from unsloth import FastLanguageModel

# Model Configuration
# Supported models: Llama, Mistral, Yi, CodeLlama
MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048  # Maximum sequence length (supports RoPE Scaling)
LOAD_IN_4BIT = True     # Load in 4-bit to reduce memory
# Auto-detection (Float16 for T4/V100 GPUs, Bfloat16 for Ampere+)
DTYPE = None  # None for auto detection.

# Load Model with Unsloth
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.9.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
  Set XFORMERS_MORE_DETAILS=1 for more details


Switching to PyTorch attention since your Xformers is broken.

Unsloth: Xformers was not installed correctly.
Please install xformers separately first.
Then confirm if it's correctly installed by running:
python -m xformers.info

Longer error message:
xFormers can't load C++/CUDA extensions. xFormers was built for:
    PyTorch 2.6.0+cu124 with CUDA 1204 (you have 2.9.0+cu126)
    Python  3.12.9 (you have 3.12.12)
  Please reinstall xformers (see https://github.com/facebookresearch/xformers#installing-xformers)
  Memory-efficient attention, SwiGLU, sparse and more won't be available.
🦥 Unsloth Zoo will now patch everything to make training faster!
Unsloth: Could not find Trainer class in trl.trainer.bco_trainer. Found: ['BCOTrainer', '_BCOTrainer']
==((====))==  Unsloth 2025.11.6: Fast Llama patching. Transformers: 4.57.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        

model.safetensors:   0%|          | 0.00/2.24G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/234 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/454 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.2M [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

## 3. LoRA Adapter Configuration

Let's add the LoRA adapters to the model. Unsloth automatically optimizes LoRA operations to maximize speed and reduce memory usage.

---

* ```lora_r```: Choose any number > 0 ! Suggested 8, 16, 32, 64, 128  
* ```target_modules```: List of model modules to apply LoRA adapters to  
* ```lora_alpha```: Scaling factor for LoRA adapters  
* ```lora_dropout```: Dropout rate for LoRA adapters (supports any value, but 0 is optimized)  
* ```lora_bias```: Bias configuration for LoRA adapters (supports any value, but "none" is optimized)

In [ ]:
# LoRA Parameters
LORA_R = 16              # Rank LoRA (higher = more trainable parameters)
LORA_ALPHA = 16          # Scaling factor for LoRA
LORA_DROPOUT = 0         # Dropout (0 is optimized by Unsloth)
LORA_BIAS = "none"       # Type of bias ("none" is optimized)
USE_GRADIENT_CHECKPOINTING = True  # Reduces memory at the cost of speed

# Target modules: all linear layers of attention and MLP
TARGET_MODULES = [
    "q_proj",    # Query projection
    "k_proj",    # Key projection
    "v_proj",    # Value projection
    "o_proj",    # Output projection
    "gate_proj",  # Gate projection (MLP)
    "up_proj",   # Up projection (MLP)
    "down_proj",  # Down projection (MLP)
]

model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    target_modules=TARGET_MODULES,
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    bias=LORA_BIAS,
    # [NEW] "unsloth" uses 30% less VRAM, fits 2x larger batch sizes!
    use_gradient_checkpointing="unsloth",
    random_state=3407,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2025.11.6 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [ ]:
# Print parameter information
trainable_params = sum(p.numel()
                       for p in model.parameters() if p.requires_grad)
all_params = sum(p.numel() for p in model.parameters())
trainable_percent = 100 * trainable_params / all_params

print(f"\n✓ LoRA adapters applied!")
print(f"Trainable parameters: {trainable_params:,} ({trainable_percent:.4f}%)")
print(f"Total parameters: {all_params:,}")

## 4. Dataset Preparation

Let's load and prepare the dataset for training. In this example, we use the Alpaca dataset.

In [ ]:
from datasets import load_dataset
dataset = load_dataset("mlabonne/FineTome-100k", split="train")

README.md:   0%|          | 0.00/982 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/117M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/100000 [00:00<?, ? examples/s]

## 5.Formatting Prompts

In [ ]:
from typing import Any, cast
from unsloth.chat_templates import get_chat_template
from unsloth.chat_templates import standardize_sharegpt

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)


def formatting_prompts_func(examples):
    convos = examples["conversations"]
    texts = [tokenizer.apply_chat_template(
        convo, tokenize=False, add_generation_prompt=False) for convo in convos]
    return {"text": texts, }


dataset = standardize_sharegpt(dataset)
dataset = dataset.map(formatting_prompts_func, batched=True,)

# Use typing.cast to satisfy static type checkers when indexing by integer
item = cast(Any, dataset[5])  # type: ignore[var-annotated]
# View the original conversation format
print(item["conversations"])
# View the same item in the formatted text format
print(item["text"])

Unsloth: Standardizing formats (num_proc=2):   0%|          | 0/100000 [00:00<?, ? examples/s]

Map:   0%|          | 0/100000 [00:00<?, ? examples/s]

[{'content': 'How do astronomers determine the original wavelength of light emitted by a celestial body at rest, which is necessary for measuring its speed using the Doppler effect?', 'role': 'user'}, {'content': 'Astronomers make use of the unique spectral fingerprints of elements found in stars. These elements emit and absorb light at specific, known wavelengths, forming an absorption spectrum. By analyzing the light received from distant stars and comparing it to the laboratory-measured spectra of these elements, astronomers can identify the shifts in these wavelengths due to the Doppler effect. The observed shift tells them the extent to which the light has been redshifted or blueshifted, thereby allowing them to calculate the speed of the star along the line of sight relative to Earth.', 'role': 'assistant'}]
<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

<|eot_id|><|start_header_id|>user<|end_header_id|

## 6. Setting Up and Configuring the Trainer

Let's configure the training parameters optimized for Unsloth and NVIDIA GPUs.

In [ ]:
from trl.trainer.sft_trainer import SFTTrainer
from transformers import TrainingArguments, DataCollatorForSeq2Seq
from unsloth import is_bfloat16_supported

OUTPUT_DIR = "./outputs"
NUM_TRAIN_EPOCHS = 3
# Batch size per device (increase if you have more memory)
PER_DEVICE_TRAIN_BATCH_SIZE = 2
# Gradient accumulation (effective batch size = 2*4 = 8)
GRADIENT_ACCUMULATION_STEPS = 4
LEARNING_RATE = 2e-4                  # Learning rate
WARMUP_STEPS = 10                     # Warmup steps
MAX_STEPS = 60                        # Maximum steps (-1 = use epochs)
LOGGING_STEPS = 10                    # Logging frequency
OPTIMIZER = "adamw_8bit"              # 8-bit optimizer to reduce memory

# Auto-detection of precision type
use_fp16 = not is_bfloat16_supported()
use_bf16 = is_bfloat16_supported()

training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,  # Directory to save model checkpoints
    per_device_train_batch_size=PER_DEVICE_TRAIN_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    warmup_steps=WARMUP_STEPS,
    # num_train_epochs = NUM_TRAIN_EPOCHS, # Set this for 1 full training run.
    max_steps=MAX_STEPS,
    learning_rate=LEARNING_RATE,
    fp16=not use_bf16,
    bf16=use_bf16,
    logging_steps=LOGGING_STEPS,
    optim=OPTIMIZER,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    seed=3407,
    report_to="none",  # Use this for WandB etc
)

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

### Initializing the Trainer

Let's create TRL's `SFTTrainer` (Supervised Fine-Tuning Trainer) that will manage the training process.

In [ ]:
trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    data_collator=DataCollatorForSeq2Seq(tokenizer=tokenizer),
    tokenizer=tokenizer,
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_num_proc=2,
    packing=False,  # Can make training 5x faster for short sequences.
    args=training_args
)

## 7. Training only on Assistant Responses

In [ ]:
from unsloth.chat_templates import train_on_responses_only
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",     # Marks user input
    # Marks assistant response
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)
# Begin training
trainer_stats = trainer.train()

Map (num_proc=6):   0%|          | 0/100000 [00:00<?, ? examples/s]

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 100,000 | Num Epochs = 1 | Total steps = 60
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 24,313,856 of 3,237,063,680 (0.75% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
1,0.777600
2,0.813400
3,1.078600
4,0.886900
5,0.762500
6,0.934300
7,0.602400
8,0.998400
9,0.878000
10,0.759200


## 8. Inference - Generating responses

In [ ]:
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(
    tokenizer,
    chat_template="llama-3.1",
)
FastLanguageModel.for_inference(model)  # Enable native 2x faster inference

messages = [
    {"role": "user", "content": "Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,"},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,  # Must add for generation
    return_tensors="pt",
).to("cuda")

outputs = model.generate(input_ids=inputs, max_new_tokens=64, use_cache=True,
                         temperature=1.5, min_p=0.1)

# Decode the generated tokens into human-readable text
text = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(text)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


system

Cutting Knowledge Date: December 2023
Today Date: 26 July 2024

user

Continue the fibonnaci sequence: 1, 1, 2, 3, 5, 8,assistant

The Fibonacci sequence is a series of numbers in which each number is the sum of the two preceding numbers. The sequence you provided starts with 1, 1, 2, 3, 5, and 8. Here are the next three numbers in the sequence:
9, 14, 23


## 9. Saving and Loading the Fine-Tuned Model

### Save Locally

In [9]:
model_name = "Llama32_fine_tuned"
model.save_pretrained(model_name)
tokenizer.save_pretrained(model_name)

('Llama32_fine_tuned/tokenizer_config.json',
 'Llama32_fine_tuned/special_tokens_map.json',
 'Llama32_fine_tuned/chat_template.jinja',
 'Llama32_fine_tuned/tokenizer.json')

### Save the Full Model in GGUF Format (Optional)

In [ ]:
model.push_to_hub_gguf(model_name, tokenizer, quantization_method="q4_k_m")

### Load the LoRA Adapters for Inference

In [ ]:
from transformers import TextStreamer
from unsloth import FastLanguageModel
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="Llama32_fine_tuned",  # Name of your fine-tuned model
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=DTYPE,
    load_in_4bit=LOAD_IN_4BIT,
)
FastLanguageModel.for_inference(model)  # Enable optimized inference

messages = [
    {"role": "user", "content": "Describe a tall tower in the capital of France."},
]
inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
).to("cuda")
text_streamer = TextStreamer(tokenizer, skip_prompt=True)
_ = model.generate(
    input_ids=inputs,
    streamer=text_streamer,
    max_new_tokens=128,
    use_cache=True,
    temperature=1.5,
    min_p=0.1
)

## 13. Metrics and Analysis

Let's analyze GPU memory usage during training.

In [ ]:
if torch.cuda.is_available():
    print("\n" + "="*50)
    print("GPU STATISTICS")
    print("="*50)

    # Allocated memory
    allocated = torch.cuda.memory_allocated(0) / 1024**3
    reserved = torch.cuda.memory_reserved(0) / 1024**3
    max_allocated = torch.cuda.max_memory_allocated(0) / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3

    print(f"\nMemory GPU usage:")
    print(f"  - Allocated: {allocated:.2f} GB")
    print(f"  - Reserved: {reserved:.2f} GB")
    print(f"  - Max Allocated: {max_allocated:.2f} GB")
    print(f"  - Total available: {total:.2f} GB")
    print(f"  - Percentage used: {(max_allocated/total)*100:.1f}%")

    # Free cache
    torch.cuda.empty_cache()
    print("\n✓ GPU cache cleared")